In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS cre_catalog_location
URL 'abfss://cre@crmdataplatform.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL cre_adls_cred);

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS cre_catalog
MANAGED LOCATION 'abfss://cre@crmdataplatform.dfs.core.windows.net/'
COMMENT 'CRE main catalog';

In [0]:
%sql
-- Landing (raw files)
CREATE SCHEMA cre_catalog.landing
MANAGED LOCATION 'abfss://cre@crmdataplatform.dfs.core.windows.net/landing/'
COMMENT 'Raw source files from NYC Tax Parcel and enrichment datasets';

-- Bronze (raw ingested tables)
CREATE SCHEMA cre_catalog.bronze
MANAGED LOCATION 'abfss://cre@crmdataplatform.dfs.core.windows.net/bronze/'
COMMENT 'Raw ingested tables, no transformation';

-- Silver (cleaned and enriched)
CREATE SCHEMA cre_catalog.silver
MANAGED LOCATION 'abfss://cre@crmdataplatform.dfs.core.windows.net/silver/'
COMMENT 'Cleaned, joined and enriched tables';

-- Gold (star schema for analytics)
CREATE SCHEMA cre_catalog.gold
MANAGED LOCATION 'abfss://cre@crmdataplatform.dfs.core.windows.net/gold/'
COMMENT 'Star schema, analytics ready'

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS cre_catalog.landing.raw_files
LOCATION 'abfss://cre@crmdataplatform.dfs.core.windows.net/landing/'
COMMENT 'Raw source files landing zone';

-- Bronze volume
CREATE VOLUME IF NOT EXISTS cre_catalog.bronze.data
COMMENT 'Bronze layer data';

-- Silver volume
CREATE VOLUME IF NOT EXISTS cre_catalog.silver.data
COMMENT 'Silver layer data';

-- Gold volume
CREATE VOLUME IF NOT EXISTS cre_catalog.gold.data
COMMENT 'Gold layer data';

In [0]:
%python
dbutils.fs.ls('/Volumes/cre_catalog/landing/raw_files/')

In [0]:
%python
# Read the TSV file
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load('/Volumes/cre_catalog/landing/raw_files/nyc_data.tsv')

# Check first few rows
display(df.limit(10))

In [0]:
# %python
# # Register as temp view
# df.createOrReplaceTempView("nyc_tax_parcels")

In [0]:
# %sql
# select 
#     PARID,
#     BORO,
#     BLOCK,
#     LOT,
#     OWNER,
#     STREET_NAME,
#     BLD_STORY,
#     NUM_BLDGS,
#     GROSS_SQFT
#    from nyc_tax_parcels where num_bldgs=2 and block = '564' limit 10;